# Antisaccade experiment — analysis with pyxations

This notebook demonstrates a full analysis pipeline for webcam eye-tracking data
using [pyxations](https://github.com/NeuroLIAA/pyxations).

**Pipeline:**
1. Convert raw jsPsych/WebGazer CSVs to BIDS format
2. Detect eye movements with REMoDNaV and propagate behavioral metadata
3. Compute error rates and reaction times per participant

## Setup

In [ ]:
import sys
sys.path.insert(0, '../../pyxations')  # remove if installed via pip

import pyxations as pyx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import ranksums

sns.set_theme(style='whitegrid', font_scale=1.2)

In [ ]:
RAW_DATA    = Path('./raw_data')
OUTPUT_DIR  = Path('.')
BIDS_NAME   = 'antisaccades_bids'
SCREEN_W    = 1366
SCREEN_H    = 768

BEHAVIORAL_COLS = [
    'typeOfSaccade', 'cueShownAtLeft',
    'itiEnd', 'fixEnd', 'intraEnd', 'visualEnd', 'responseEnd',
    'viewportWidth', 'isTutorial',
]

## Step 1 — Convert to BIDS

In [ ]:
bids_path = pyx.dataset_to_bids(
    OUTPUT_DIR, RAW_DATA, BIDS_NAME, format_name='webgazer'
)
print(f'BIDS dataset: {bids_path}')

## Step 2 — Eye movement detection + behavioral metadata

In [ ]:
pyx.compute_derivatives_for_dataset(
    bids_path,
    dataset_format='webgazer',
    detection_algorithm='remodnav',
    overwrite=True,
    screen_height=SCREEN_H,
    screen_width=SCREEN_W,
    behavioral_columns=BEHAVIORAL_COLS,
)

Each session now has:
- `samples.feather` — gaze samples with behavioral columns joined by `trial_index`
- `remodnav_events/` — fixations, saccades and blinks detected by REMoDNaV
- `events.tsv` — BIDS-compatible trial table

## Step 3 — Error rates and RTs per participant

For each trial we determine correctness from gaze position during the response window
(`visualEnd` → `responseEnd`, i.e. the 650 ms after the peripheral target appears).
A trial is correct when the mean gaze X is on the expected side of screen center:
- **Prosaccade**: toward the cue
- **Antisaccade**: away from the cue

In [ ]:
deriv_path = bids_path.with_name(bids_path.name + '_derivatives')

rows = []

for session_dir in sorted((deriv_path / 'sub-0001').iterdir()):
    samples_file = session_dir / 'samples.feather'
    if not samples_file.exists():
        continue

    subject_id = session_dir.name.replace('ses-', '')
    df = pd.read_feather(samples_file)

    # Keep only real saccade trials (exclude tutorial and non-saccade rows)
    df = df[
        df['typeOfSaccade'].notna() &
        (df['isTutorial'] != True)
    ].copy()

    if df.empty:
        continue

    # Use the most common viewport width as screen center reference
    screen_center = df['viewportWidth'].median() / 2

    for trial_id, trial in df.groupby('trial_index'):
        visual_end  = trial['visualEnd'].iloc[0]
        response_end = trial['responseEnd'].iloc[0]

        # Samples during response window (tSample is within-trial time in ms)
        response_window = trial[
            (trial['tSample'] >= visual_end) &
            (trial['tSample'] <= response_end)
        ]

        if response_window.empty:
            continue

        mean_x         = response_window['X'].mean()
        cue_at_left    = bool(trial['cueShownAtLeft'].iloc[0])
        saccade_type   = trial['typeOfSaccade'].iloc[0]

        gaze_on_cue_side = (mean_x < screen_center) == cue_at_left

        if saccade_type == 'prosaccade':
            correct = gaze_on_cue_side
        else:
            correct = not gaze_on_cue_side

        rows.append({
            'subject':      subject_id,
            'trial_index':  trial_id,
            'saccade_type': saccade_type,
            'correct':      correct,
        })

df_trials = pd.DataFrame(rows)
print(f'{len(df_trials)} trials across {df_trials["subject"].nunique()} subjects')
df_trials.head()

In [ ]:
df_summary = (
    df_trials
    .groupby(['subject', 'saccade_type'])['correct']
    .agg(error_rate=lambda x: 1 - x.mean(), n_trials='count')
    .reset_index()
)

df_summary.head(6)

## Step 4 — Plot and statistics

In [ ]:
fig, ax = plt.subplots(figsize=(4, 5))

sns.boxplot(
    data=df_summary,
    x='saccade_type', y='error_rate',
    order=['prosaccade', 'antisaccade'],
    width=0.4, color='white',
    ax=ax,
)
sns.stripplot(
    data=df_summary,
    x='saccade_type', y='error_rate',
    order=['prosaccade', 'antisaccade'],
    color='steelblue', alpha=0.6, jitter=True,
    ax=ax,
)

ax.set_ylabel('Error rate')
ax.set_xlabel('')
ax.set_xticklabels(['Prosaccade', 'Antisaccade'])
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('./result_plots/error_rates_pyxations.png', dpi=150)
plt.show()

In [ ]:
pro  = df_summary.query("saccade_type == 'prosaccade'")['error_rate']
anti = df_summary.query("saccade_type == 'antisaccade'")['error_rate']

stat, p = ranksums(pro, anti)

print('Error rate (mean ± SD)')
print(f'  Prosaccade:  {pro.mean():.3f} ± {pro.std():.3f}')
print(f'  Antisaccade: {anti.mean():.3f} ± {anti.std():.3f}')
print(f'  Wilcoxon rank-sum: W={stat:.2f}, p={p:.4f}')